In [1]:
train_set = '../data/raw/sample_23.csv' # do not touch
test_set = '../data/raw/sample_24.csv' # do not touch

predictions_weight = {
    # '../data/predictions/sklearn_forecast_2024_cluster_specific_combined_case5_clusters.csv': 0.9,
    # '../data/predictions/flo/pred_2024_cluster_xgb(6).csv': 0.1,
    '../data/predictions/pred_2024_global_xgb.csv': 1,
    # '../data/predictions/flo_static_shapelets/pred_2024_cluster_xgb(7).csv': 0.05,
}

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error


def _resolve_path(path_like):
    path = Path(path_like)
    if path.is_absolute():
        return path
    return (Path.cwd() / path).resolve()


def _pick_id_col(dfs):
    preferred = ["ID", "id", "Id", "series_id", "item_id"]
    common_cols = set(dfs[0].columns)
    for df in dfs[1:]:
        common_cols &= set(df.columns)
    for col in preferred:
        if col in common_cols:
            return col
    for col in dfs[0].columns:
        if col in common_cols:
            return col
    raise ValueError("No shared identifier column found across the prediction files.")


def _pick_value_cols(dfs, id_col):
    common_cols = [c for c in dfs[0].columns if c != id_col]
    for df in dfs[1:]:
        common_cols = [c for c in common_cols if c in df.columns and c != id_col]
    if not common_cols:
        raise ValueError("No shared value columns found across the prediction files.")
    return common_cols


prediction_items = list(predictions_weight.items())
if not prediction_items:
    raise ValueError("predictions_weight must contain at least one prediction file.")

prediction_frames = []
for path, weight in prediction_items:
    df_pred = pd.read_csv(_resolve_path(path))
    prediction_frames.append((path, weight, df_pred))

dfs_only = [df for _, _, df in prediction_frames]
id_col = _pick_id_col(dfs_only)
value_cols = _pick_value_cols(dfs_only, id_col)

common_ids = set(dfs_only[0][id_col].dropna())
for df in dfs_only[1:]:
    common_ids &= set(df[id_col].dropna())
if not common_ids:
    raise ValueError("No overlapping IDs found across the prediction files.")
common_ids = sorted(common_ids)

weighted_sum = None
total_weight = 0.0

for path, weight, df_pred in prediction_frames:
    frame = df_pred[df_pred[id_col].isin(common_ids)].copy()
    frame = frame[[id_col] + value_cols].set_index(id_col)
    frame = frame.apply(pd.to_numeric, errors="coerce")
    frame = frame.loc[common_ids]
    if weighted_sum is None:
        weighted_sum = frame * weight
    else:
        weighted_sum = weighted_sum.add(frame * weight, fill_value=0.0)
    total_weight += weight

if total_weight == 0:
    raise ValueError("Total ensemble weight is zero.")

blended_values = weighted_sum / total_weight
blended = blended_values.reset_index()

output_path = _resolve_path("../data/predictions/weighted_average_predictions.csv")
blended.to_csv(output_path, index=False)

test_path = _resolve_path(test_set)
df_true = pd.read_csv(test_path)
if id_col not in df_true.columns:
    raise ValueError(f"Label file does not contain the identifier column '{id_col}'.")

label_value_cols = [c for c in value_cols if c in df_true.columns]
if not label_value_cols:
    raise ValueError("No overlapping value columns found between labels and blended predictions.")

merged = df_true[[id_col] + label_value_cols].merge(
    blended[[id_col] + label_value_cols],
    on=id_col,
    how="inner",
    suffixes=("_true", "_pred"),
)
if merged.empty:
    raise ValueError("No overlapping IDs between the labels and blended predictions.")

y_true = merged[[f"{c}_true" for c in label_value_cols]].to_numpy().ravel()
y_pred = merged[[f"{c}_pred" for c in label_value_cols]].to_numpy().ravel()
mae = mean_absolute_error(y_true, y_pred)

print(f"Saved: {output_path}")
print(f"MAE: {mae:.6f}")
display(blended.head())

Saved: /home/gopes/Documents/Clustering-And-Forecasting-TimeSeries-PlayingGround/data/predictions/weighted_average_predictions.csv
MAE: 4.827977


,ID,2024-01-01,2024-01-02,2024-01-03,2024-01-04,2024-01-05,2024-01-06,2024-01-07,2024-01-08,2024-01-09,...,2024-12-22,2024-12-23,2024-12-24,2024-12-25,2024-12-26,2024-12-27,2024-12-28,2024-12-29,2024-12-30,2024-12-31
0,22,10.283529,10.092375,10.021766,10.268026,9.520789,10.523973,11.159896,10.512636,10.570364,...,10.926749,10.701824,10.768353,10.749112,10.715153,10.786255,10.804290,10.928328,10.692465,10.768936
1,42,42.575470,38.261032,40.081985,41.466137,41.017937,42.523390,45.420525,43.506714,41.653614,...,37.742535,37.617786,37.536674,37.523740,37.553436,37.520096,37.657760,37.746162,37.636124,37.552456
2,56,5.862229,5.440917,4.839586,5.298032,6.248977,7.596391,7.753248,6.715525,6.266632,...,5.751427,5.509971,5.528311,5.532347,5.510874,5.495715,5.604911,5.728446,5.510612,5.510029
3,58,14.477797,15.525618,16.583883,13.453336,14.127836,15.115213,15.708204,14.278194,14.352284,...,16.345650,16.210964,16.280329,16.218897,16.178833,16.175539,16.298580,16.321300,16.240747,16.214937
4,64,2.383181,2.344875,2.349334,2.380768,2.393020,2.386398,2.358158,2.397314,2.358540,...,1.920808,1.925666,1.924531,1.925187,1.922745,1.922859,1.922238,1.920808,1.925666,1.924531
